In [ ]:
import os
import random
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

IMG_SIZE   = 224         
PATCH_SIZE = 56          
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2
BATCH_SIZE  = 8
MAX_PER_CLASS = 8000

# Paths
DATA_ROOT = "/workspace/OCT2017"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")


CLASS_NAMES = sorted([
    d for d in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, d))
])
NUM_CLASSES = len(CLASS_NAMES)

print("Classes:", CLASS_NAMES)
print("NUM_CLASSES:", NUM_CLASSES)


In [ ]:
clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(4,4))

def apply_clahe_np(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    L, A, B = cv2.split(lab)
    L2 = clahe.apply(L)
    lab = cv2.merge((L2, A, B))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def load_and_patch_np(path, label):
    path = path.decode()

    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = apply_clahe_np(img)

    img = img.astype("float32") / 255.0

    patches = []
    for i in range(0, IMG_SIZE, PATCH_SIZE):
        for j in range(0, IMG_SIZE, PATCH_SIZE):
            patch = img[i:i+PATCH_SIZE, j:j+PATCH_SIZE, :]
            patches.append(patch)

    return np.stack(patches, axis=0), label


In [ ]:
def tf_wrapper(path, label):
    patches, lab = tf.numpy_function(
        load_and_patch_np,
        [path, label],
        [tf.float32, tf.int32]
    )
    patches.set_shape((NUM_PATCHES, PATCH_SIZE, PATCH_SIZE, 3))
    lab.set_shape(())
    return patches, lab


In [ ]:
def build_file_list(root_dir, max_per_class=None):
    filepaths, labels = [], []
    for idx, cname in enumerate(CLASS_NAMES):
        folder = os.path.join(root_dir, cname)

        imgs = [
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        if max_per_class and len(imgs) > max_per_class:
            random.shuffle(imgs)
            imgs = imgs[:max_per_class]

        filepaths.extend(imgs)
        labels.extend([idx] * len(imgs))

    return filepaths, labels


def make_dataset(root_dir, shuffle=False, max_per_class=None):
    filepaths, labels = build_file_list(root_dir, max_per_class)

    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))

    if shuffle:
        ds = ds.shuffle(len(filepaths), seed=SEED)

    ds = ds.map(tf_wrapper, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(TRAIN_DIR, shuffle=True,  max_per_class=MAX_PER_CLASS)
val_ds   = make_dataset(VAL_DIR)
test_ds  = make_dataset(TEST_DIR)

len(list(train_ds)), len(list(val_ds)), len(list(test_ds))


In [ ]:
class GroupNorm(layers.Layer):
    def __init__(self, groups=8, epsilon=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.groups = groups
        self.epsilon = epsilon

    def build(self, input_shape):
        C = int(input_shape[-1])
        self.groups = min(self.groups, C)

        self.gamma = self.add_weight(
            name="gamma",
            shape=(C,),
            initializer="ones",
            trainable=True
        )
        self.beta = self.add_weight(
            name="beta",
            shape=(C,),
            initializer="zeros",
            trainable=True
        )

    def call(self, x):
        N, H, W, C = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], tf.shape(x)[3]
        G = self.groups

        x = tf.reshape(x, [N, H, W, G, C // G])
        mean, var = tf.nn.moments(x, axes=[1,2,4], keepdims=True)
        x = (x - mean) / tf.sqrt(var + self.epsilon)
        x = tf.reshape(x, [N, H, W, C])

        return x * self.gamma + self.beta


In [ ]:
def residual_block(x, filters, stride=1, groups=8, name=None):
    shortcut = x

    y = layers.Conv2D(filters, 3, strides=stride, padding="same",
                      use_bias=False, name=f"{name}_conv1")(x)
    y = GroupNorm(groups=groups, name=f"{name}_gn1")(y)
    y = layers.ReLU()(y)

    y = layers.Conv2D(filters, 3, padding="same", use_bias=False,
                      name=f"{name}_conv2")(y)
    y = GroupNorm(groups=groups, name=f"{name}_gn2")(y)

    if shortcut.shape[-1] != filters or stride != 1:
        shortcut = layers.Conv2D(filters, 1, strides=stride,
                                 padding="same", use_bias=False,
                                 name=f"{name}_proj")(shortcut)
        shortcut = GroupNorm(groups=groups, name=f"{name}_proj_gn")(shortcut)

    out = layers.Add()([y, shortcut])
    return layers.ReLU()(out)


In [ ]:
def build_patch_cnn():
    inp = layers.Input((PATCH_SIZE, PATCH_SIZE, 3), name="patch_input")

    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    x = GroupNorm(groups=8)(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)

    x = residual_block(x, 32, name="res1_1")
    x = residual_block(x, 32, name="res1_2")

    x = residual_block(x, 64, stride=2, name="res2_1")
    x = residual_block(x, 64, name="res2_2")

    x = residual_block(x, 128, stride=2, name="res3_1")
    x = residual_block(x, 128, name="res3_2")

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation="relu")(x)

    return models.Model(inp, x, name="PatchCNN")


patch_cnn = build_patch_cnn()
patch_cnn.summary()


In [ ]:
class TopKPooling(layers.Layer):
    def __init__(self, k=3, **kwargs):
        super().__init__(**kwargs)
        self.k = k

    def call(self, x):
        scores = tf.reduce_max(x, axis=-1) 

        topk_idx = tf.argsort(scores, direction="DESCENDING")[:, :self.k]  


        batch = tf.range(tf.shape(x)[0])[:, None]    
        batch = tf.tile(batch, [1, self.k])         

        gather_idx = tf.stack([batch, topk_idx], axis=-1) 

        selected = tf.gather_nd(x, gather_idx) 

        return tf.reduce_mean(selected, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])


In [ ]:
inputs = layers.Input(
    shape=(NUM_PATCHES, PATCH_SIZE, PATCH_SIZE, 3),
    name="patch_bag"
)

x = layers.TimeDistributed(patch_cnn)(inputs)
x = TopKPooling(k=3)(x)

x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

patch_model = models.Model(inputs, outputs, name="Patch_MIL_Model")
patch_model.summary()


In [ ]:
patch_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("best_patch_model.keras", save_best_only=True)
]

history = patch_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)


In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Accuracy")
plt.legend(["Train", "Val"])

plt.subplot(1,2,2)
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("Loss")
plt.legend(["Train", "Val"])

plt.show()


In [ ]:
test_loss, test_acc = patch_model.evaluate(test_ds)
print("TEST ACCURACY:", test_acc)

y_true, y_pred = [], []

for patches, labels in test_ds:
    p = patch_model.predict(patches, verbose=0)
    y_pred.extend(np.argmax(p, axis=1))
    y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
def visualize_topk(scan_path, k=3):
    patches, _ = load_and_patch_np(scan_path.encode(), 0)

    feats = patch_cnn.predict(patches, verbose=0)
    feats = feats.reshape(1, NUM_PATCHES, -1)

    scores = tf.reduce_max(feats, axis=-1)[0]
    topk_idx = tf.math.top_k(scores, k=k).indices.numpy()

    fig, axes = plt.subplots(4,4, figsize=(6,6))
    t = 0
    for i in range(4):
        for j in range(4):
            ax = axes[i][j]
            ax.imshow(patches[t])
            if t in topk_idx:
                ax.set_title("TOP-K", color="red")
            ax.axis("off")
            t += 1

    plt.suptitle("Top-K Patches")
    plt.tight_layout()
    plt.show()


example_path = os.path.join(TEST_DIR, CLASS_NAMES[0],
                            os.listdir(os.path.join(TEST_DIR, CLASS_NAMES[0]))[0])

visualize_topk(example_path)
